In [28]:
### rename for unrelaxed
import pandas as pd
import os
# complex names
csv_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/newBM_33.csv"
df = pd.read_csv(csv_path)

complex_list=df["pdb_wchains"].to_list()
for complex_name in complex_list:
    folder_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm/"+str(complex_name)+"_esm/"
    prefix = str(complex_name)+"_esm_"  # prefix from af2

    for filename in os.listdir(folder_path):
        if filename.endswith(".pdb"):  # 
            old_path = os.path.join(folder_path, filename)
            new_filename = prefix + filename 
            new_path = os.path.join(folder_path, new_filename)
            
            os.rename(old_path, new_path)  
            print(f"Renamed: {filename} → {new_filename}")
    
    print("renaming complete")

Renamed: ptm0.398_r3_default.pdb → 7r58_H_L_A_esm_ptm0.398_r3_default.pdb
renaming complete
Renamed: ptm0.422_r3_default.pdb → 7sjs_H_L_A_esm_ptm0.422_r3_default.pdb
renaming complete
Renamed: ptm0.570_r3_default.pdb → 7so5_H_L_A_esm_ptm0.570_r3_default.pdb
renaming complete
Renamed: ptm0.483_r3_default.pdb → 7tp4_H_L_A_esm_ptm0.483_r3_default.pdb
renaming complete
Renamed: ptm0.420_r3_default.pdb → 7u2d_H_L_A_esm_ptm0.420_r3_default.pdb
renaming complete
Renamed: ptm0.578_r3_default.pdb → 7u5b_H_L_A_esm_ptm0.578_r3_default.pdb
renaming complete
Renamed: ptm0.522_r3_default.pdb → 7uvf_H_L_A_esm_ptm0.522_r3_default.pdb
renaming complete
Renamed: ptm0.272_r3_default.pdb → 7uz7_H_L_A_esm_ptm0.272_r3_default.pdb
renaming complete
Renamed: ptm0.278_r3_default.pdb → 7uz8_H_L_A_esm_ptm0.278_r3_default.pdb
renaming complete
Renamed: ptm0.413_r3_default.pdb → 7vaz_H_L_A_esm_ptm0.413_r3_default.pdb
renaming complete
Renamed: ptm0.272_r3_default.pdb → 7woa_H_L_A_esm_ptm0.272_r3_default.pdb
renami

In [35]:
## dockq Functions

import os
import sys
import csv
import glob

from DockQ.DockQ import load_PDB, run_on_all_native_interfaces
from statistics import mean

def merge_chains(model, chains_to_merge):
    print(f"Merging chains {chains_to_merge} in model")
    for chain in chains_to_merge[1:]:
        for res in list(model[chain]):
            res.id = (chains_to_merge[0], res.id[1], res.id[2])
            model[chains_to_merge[0]].add(res)
        model.detach_child(chain)
    model[chains_to_merge[0]].id = "".join(chains_to_merge)
    return model

def calculate_dockq(model, native, chain_map):
    print(f"Calculating DockQ score with chain_map: {chain_map}")
    results, dockq_score = run_on_all_native_interfaces(model, native, chain_map=chain_map)
    return results, dockq_score

def process_models(models):
    results_list = []
    for model_file, native_file in models:
        print(f"Processing model: {model_file}, native: {native_file}")
        model_id = os.path.basename(model_file).split(".")[0]
        model = load_PDB(model_file)
        native = load_PDB(native_file)

        chain_ids = list(model.child_dict.keys())
        print(chain_ids)
        native_chain_ids = list(native.child_dict.keys())

        if len(chain_ids) == 3:
            print(f"Model {model_id} has 3 chains: {chain_ids}")
            
            # Merge A and B chains and recalculate
            model_merged = merge_chains(model, chain_ids[:2])
            print(model_merged)
            native_merged = merge_chains(native, native_chain_ids[:2])
            chain_map_merged = {native_chain_ids[2]: chain_ids[2], "".join(native_chain_ids[:2]): "".join(chain_ids[:2])}
            results_merged, dockq_score_merged = calculate_dockq(model_merged, native_merged, chain_map_merged)
            merged_result = results_merged[list(results_merged.keys())[0]]
            results_list.append((model_id, merged_result['DockQ'], merged_result['fnat'],
                                merged_result['iRMSD'], merged_result['LRMSD'],merged_result['F1']))

        elif len(chain_ids) == 2:
            print(f"Model {model_id} has 2 chains: {chain_ids}")
            chain_map = {native_chain_ids[0]: chain_ids[0], native_chain_ids[1]: chain_ids[1]}
            results, dockq_score = calculate_dockq(model, native, chain_map)
            results_list.append((model_id, results[list(results.keys())[0]]['DockQ'], results[list(results.keys())[0]]['fnat'],
                                results[list(results.keys())[0]]['iRMSD'], results[list(results.keys())[0]]['LRMSD'],
                                results[list(results.keys())[0]]['F1']))

        else:
            print(f"Model {model_id} does not have 2 or 3 chains: {chain_ids}, skipping.")
            continue

    return results_list

def save_results_to_csv(results, filename):
    print(f"Saving results to CSV file: {filename}")
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['model_id', 'DockQ', 'fnat',"iRMSD","LRMSD","F1"])

        for model_id, dockq, fnat, irms, lrms, f1 in results:
            print(f"Writing row: {model_id}, {dockq}, {fnat}, {irms},{lrms},{f1}")
            writer.writerow([model_id, dockq, fnat, irms, lrms,f1])
    
def main(directory, original_directory):
    pdb_files = glob.glob(f"{directory}/*_esm*.pdb")
    if not pdb_files:
        print(f"No PDB files found in directory: {directory}")
        return

    print(f"Found PDB files: {pdb_files}")
    models = []

    for pdb_file in pdb_files:
        pdb_id_chains = os.path.basename(pdb_file).split('_esm')[0] # for af2.3
        print(f"Searching for original files for: {pdb_id_chains}")
        original_pdb_files = glob.glob(f"{original_directory}/{pdb_id_chains}.pdb")  # Find matching original PDB files
        print(f"Found original PDB files: {original_pdb_files} for {pdb_file}")
        if not original_pdb_files:
            print(f"No matching original PDB file found for {pdb_file}, skipping.")
            continue
        for original_pdb_file in original_pdb_files:
            models.append((pdb_file, original_pdb_file))
            print(f"Adding model-native pair: {pdb_file}, {original_pdb_file}")

    if not models:
        print("No valid model-native pairs found. Exiting.")
        return

    results = process_models(models)
    save_results_to_csv(results, str(pdb_id_chains)+'_'+str(method)+'_dockq_fnat_scores.csv')

In [36]:
#### dockq and fnat calculation running
import pandas as pd
import glob
import os

csv_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/newBM_33.csv"
df = pd.read_csv(csv_path)

complex_list=df["pdb_wchains"].to_list()
preds = ["esm"]
# method = preds[4]
for method in preds:
    os.chdir("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/"+str(method))
    
    # dockq functions 
    for i in complex_list:
        fpath = f"/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/{method}/{i}_{method}/"
        directory = os.path.expanduser(fpath)
        original_directory = os.path.expanduser("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/original_pdbs")
        main(directory, original_directory)
    
    
    def combine_csv_files(folder_path, output_file=None):
        # Find all CSV files in the folder
        csv_files = glob.glob(os.path.join(folder_path, "*dockq_fnat_scores.csv"))
        
        # Read and combine all CSVs
        df_list = [pd.read_csv(file) for file in csv_files]
        combined_df = pd.concat(df_list, ignore_index=True)
        
        # Save to CSV if an output file is provided
        if output_file:
            combined_df.to_csv(output_file, index=False)
            print(f"Combined CSV saved to {output_file}")
        
        return combined_df
    
    # Example usage
    folder_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/"+str(method)  
    output_csv = str(method)+"_dockq_fnat_scores.csv"  
    combined_df = combine_csv_files(folder_path, output_csv)

Found PDB files: ['/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm/7r58_H_L_A_esm/7r58_H_L_A_esm_ptm0.398_r3_default.pdb']
Searching for original files for: 7r58_H_L_A
Found original PDB files: ['/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/original_pdbs/7r58_H_L_A.pdb'] for /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm/7r58_H_L_A_esm/7r58_H_L_A_esm_ptm0.398_r3_default.pdb
Adding model-native pair: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm/7r58_H_L_A_esm/7r58_H_L_A_esm_ptm0.398_r3_default.pdb, /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/original_pdbs/7r58_H_L_A.pdb
Processing model: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm/7r58_H_L_A_esm/7r58_H_L_A_esm_ptm0.398_r3_default.pdb, native: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/original_pdbs/7r58_H_L_A.pdb
['A', 'B', 'C']
Model 7r58_H_L_A_esm_ptm0 has 3 chains: ['A', 'B', 'C']
Merging chains ['A', 'B'] in model
<Model id=/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/

In [30]:
import json
import numpy as np

import pandas as pd
import glob
import os

csv_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/newBM_33.csv"
df = pd.read_csv(csv_path)

complex_list=df["pdb_wchains"].to_list()

os.chdir("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm")

for f in complex_list:
    os.chdir(f"/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm/{f}_esm")
# File paths
    txt_file = glob.glob("*pae.txt")
    json_file = "converted_pae.json"

    # Read the PAE data from the text file
    pae_data = []
    with open(txt_file[0], "r") as f:
        for line in f:
            values = list(map(float, line.strip().split()))
            pae_data.append(values)
    
    pae_json = {"pae": pae_data}
    
    with open(json_file, "w") as f:
        json.dump(pae_json, f, indent=4)
    
    print(f"Converted JSON saved to: {json_file}")


Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: converted_pae.json
Converted JSON saved to: convert

In [2]:
#pdockq2 funcs
from Bio.PDB import PDBIO
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.Selection import unfold_entities

import numpy as np
import sys,os
import argparse
import pickle
import itertools
import pandas as pd
from scipy.optimize import curve_fit

def retrieve_IFplddt(structure, chain1, chain2_lst, max_dist):
    ## generate a dict to save IF_res_id
    chain_lst = list(chain1) + chain2_lst

    ifplddt = []
    contact_chain_lst = []
    for res1 in structure[0][chain1]:
        for chain2 in chain2_lst:
            count = 0
            for res2 in structure[0][chain2]:

                if res1.has_id('CA') and res2.has_id('CA'):
                   dis = abs(res1['CA']-res2['CA'])
                   ## add criteria to filter out disorder res
                   if dis <= max_dist:
                      ifplddt.append(res1['CA'].get_bfactor())
                      count += 1

                elif res1.has_id('CB') and res2.has_id('CB'):
                   dis = abs(res1['CB']-res2['CB'])
                   if dis <= max_dist:
                      ifplddt.append(res1['CB'].get_bfactor())
                      count += 1
            if count > 0:
              contact_chain_lst.append(chain2)
    contact_chain_lst = sorted(list(set(contact_chain_lst)))   

    if len(ifplddt)>0:
       IF_plddt_avg = np.mean(ifplddt)
    else:
       IF_plddt_avg = 0
    # print(IF_plddt_avg)

    return IF_plddt_avg, contact_chain_lst


def retrieve_IFPAEinter(structure, paeMat, contact_lst, max_dist):
    chain_lst = [x.id for x in structure[0]]
    seqlen = [len(x) for x in structure[0]]
    ifch1_col=[]
    ifch2_col=[]
    ch1_lst=[]
    ch2_lst=[]
    ifpae_avg = []
    d=10
    for ch1_idx in range(len(chain_lst)):
      idx = chain_lst.index(chain_lst[ch1_idx])
      ch1_sta=sum(seqlen[:idx])
      ch1_end=ch1_sta+seqlen[idx]
      ifpae_col = []   
      ## for each chain that shares an interface with chain1, retrieve the PAE matrix for the specific part.
      for contact_ch in contact_lst[ch1_idx]:
        index = chain_lst.index(contact_ch)
        ch_sta = sum(seqlen[:index])
        ch_end = ch_sta+seqlen[index]
        paeMat = np.array(paeMat)

        remain_paeMatrix = paeMat[ch1_sta:ch1_end,ch_sta:ch_end]

        mat_x = -1
        for res1 in structure[0][chain_lst[ch1_idx]]:
          mat_x += 1
          mat_y = -1
          for res2 in structure[0][contact_ch]:
              mat_y+=1
              if res1['CA'] - res2['CA'] <=max_dist:
                 ifpae_col.append(remain_paeMatrix[mat_x,mat_y])
      ## normalize by d(10A) first and then get the average
      if not ifpae_col:
        ifpae_avg.append(0)
      else:
        norm_if_interpae=np.mean(1/(1+(np.array(ifpae_col)/d)**2))
        ifpae_avg.append(norm_if_interpae)

    return ifpae_avg

def calc_pmidockq(ifpae_norm, ifplddt):
    df = pd.DataFrame()
    df['ifpae_norm'] = ifpae_norm
    df['ifplddt'] = ifplddt

    df['prot'] = df.ifpae_norm*df.ifplddt
    fitpopt = [1.31034849e+00, 8.47326239e+01, 7.47157696e-02, 5.01886443e-03] ## from orignal pdcokq2 fit  
    df['pmidockq'] = sigmoid(df.prot.values, *fitpopt)
    return df

def sigmoid(x, L ,x0, k, b):
    y = L / (1 + np.exp(-k*(x-x0)))+b
    return (y)

def process_pdb_file(pdb_file, json_file, distance, file_id, chains_part="", pae=None):
    
    # Parse the PDB file
    pdbp = PDBParser(QUIET=True)
    structure = pdbp.get_structure('', pdb_file)
    chains = [chain.id for chain in structure[0]]

    remain_contact_lst = []
    plddt_lst = []

    for idx in range(len(chains)):
        chain2_lst = list(set(chains)-set(chains[idx]))
        IF_plddt, contact_lst = retrieve_IFplddt(structure, chains[idx], chain2_lst, distance)
        plddt_lst.append(IF_plddt)
        remain_contact_lst.append(contact_lst)
    
    avgif_pae = retrieve_IFPAEinter(structure, pae, remain_contact_lst, distance)

    res = calc_pmidockq(avgif_pae, plddt_lst)
    pdb_id = os.path.basename(pdb_file).split('_')[0]
    
    result = {
        "model_id": pdb_file,
        "pdb_id": file_id,
        "Pdb_id_with_chains": '{0}_{1}'.format(pdb_id, chains_part),
        "ifpae_norm_ag": res['ifpae_norm'].tolist()[-1],  #  all values
        "ifpae_norm_avg": np.mean(res['ifpae_norm']),  #  average
        "ifplddt_ag": res['ifplddt'].tolist()[-1],  #  all values
        "ifplddt_avg": np.mean(res['ifplddt']),  #  average
        "PdockQ2 Antigen": res['pmidockq'].tolist()[-1], #original - antigen score
        "PdockQ2_avg": np.mean(res['pmidockq']),  #  average
    }
    return result

In [3]:
import os
import glob
import json
import pandas as pd

def find_matching_json(pdb_file, directory):
   
    json_files = glob.glob(os.path.join(directory, "converted_pae.json"))
    
    print(f"Found JSON files: {json_files}")
    
    return json_files[0] if json_files else None  
    
def run_processing(directory):
    pdb_files = glob.glob(os.path.join(directory, "*_esm*.pdb"))
    print(f"Found PDB files: {pdb_files}")  # Debugging line
    
    results = []

    for pdb_file in pdb_files:
        pdb_file_basename = os.path.basename(pdb_file)
        parts = pdb_file_basename.split('_')
        pdb_id = parts[0]
        chains_part = "_".join(parts[1:parts.index("esm")])  

        json_file_name = find_matching_json(pdb_file, directory)
        
        if json_file_name and os.path.exists(json_file_name):
            with open(json_file_name, 'r') as json_file:
                json_data = json.load(json_file)
                pae = json_data.get('pae', None)  
            
            if pae is not None:
                file_id = f"{pdb_id}_{chains_part}"

                result = process_pdb_file(pdb_file, json_file_name, 8, pdb_id, chains_part, pae)  
                print(f"Result for {pdb_file}: {result}")
                results.append(result)
            else:
                print(f"PAE not found in JSON file for {pdb_file}.")
        else:
            print(f"JSON file for {pdb_file} not found.")
    
    # Save results to CSV
    if results:
        df = pd.DataFrame(results)
        csv_file_path = f"{file_id}_{str(method)}_pdockq2_fit.csv"  
        df.to_csv(csv_file_path, index=False)
        print(f"Data saved to {csv_file_path}")
    else:
        print("No results to save.")


In [6]:
# Set directory paths
import pandas as pd
import glob
import os

csv_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/newBM_33.csv"
df = pd.read_csv(csv_path)

complex_list=df["pdb_wchains"].to_list()
preds = ["esm"]
for method in preds:
    os.chdir("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/"+str(method))
    for i in complex_list:
        fpath = f"/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/{method}/{i}_{method}"
        directory = fpath
        run_processing(directory)   
    
    def combine_csv_files(folder_path, output_file=None):
        csv_files = glob.glob(os.path.join(folder_path, "*_pdockq2_fit.csv"))

        df_list = [pd.read_csv(file) for file in csv_files]
        combined_df = pd.concat(df_list, ignore_index=True)
        if output_file:
            combined_df.to_csv(output_file, index=False)
            print(f"Combined CSV saved to {output_file}")
        return combined_df

    folder_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/"+str(method)  
    output_csv = str(method)+"_pdockq2_fit.csv"  
    combined_df = combine_csv_files(folder_path, output_csv)

Found PDB files: ['/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/esm/7r58_H_L_A_esm/7r58_H_L_A_esm_ptm0.398_r3_default.pdb']
Found JSON files: ['/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/esm/7r58_H_L_A_esm/converted_pae.json']
Result for /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/esm/7r58_H_L_A_esm/7r58_H_L_A_esm_ptm0.398_r3_default.pdb: {'model_id': '/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/esm/7r58_H_L_A_esm/7r58_H_L_A_esm_ptm0.398_r3_default.pdb', 'pdb_id': '7r58', 'Pdb_id_with_chains': '7r58_H_L_A', 'ifpae_norm_ag': 0.13791231170658355, 'ifpae_norm_avg': 0.1292551302979604, 'ifplddt_ag': 75.476, 'ifplddt_avg': 77.03303519668736, 'PdockQ2 Antigen': 0.01007720800710124, 'PdockQ2_avg': 0.009907840237906175}
Data saved to 7r58_H_L_A_esm_pdockq2_fit.csv
Found PDB files: ['/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/esm/7sjs_H_L_A_esm/7sjs_H_L_A_esm_ptm0.422_r3_default.pdb']


In [26]:
import os
import pandas as pd
os.chdir("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/esm")  
# !ls *csv
df2 = pd.read_csv('esm_dockq_fnat_scores.csv')
df = pd.read_csv('esm_pdockq2_fit.csv')
df3 = pd.read_csv('esm_metrics_25models.csv')
df2["model_id"] = df2["model_id"].str.replace("_esm_ptm0","")
df["model_id"] =df["Pdb_id_with_chains"]
df4 = pd.merge(df,df2,on="model_id")
# df4
df5 = pd.merge(df4,df3,on="model_id")
df5=df5.drop_duplicates()
df5.to_csv("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/210_recycle20/esm_pdockq2_dockq.csv")
display(df)
display(df2)
display(df3)
df5

,model_id,pdb_id,Pdb_id_with_chains,ifpae_norm_ag,ifpae_norm_avg,ifplddt_ag,ifplddt_avg,PdockQ2 Antigen,PdockQ2_avg
0,7m1h_G_A,7m1h,7m1h_G_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348
1,8d9y_H_L_A,8d9y,8d9y_H_L_A,0.000000,0.623355,0.000000,56.687153,0.007348,0.358550
2,7tcq_H_L_A,7tcq,7tcq_H_L_A,0.000000,0.624447,0.000000,56.916263,0.007348,0.366035
3,7vaz_H_L_A,7vaz,7vaz_H_L_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348
4,8cyj_H_D,8cyj,8cyj_H_D,0.128239,0.129736,25.703333,37.791667,0.007997,0.008405
...,...,...,...,...,...,...,...,...,...
204,7xy8_H_L_A,7xy8,7xy8_H_L_A,0.000000,0.212582,0.000000,56.360631,0.007348,0.017305
205,7l6v_D_A,7l6v,7l6v_D_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348
206,7l6v_F_A,7l6v,7l6v_F_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348
207,8tbq_H_L_A,8tbq,8tbq_H_L_A,0.000000,0.083764,0.000000,50.066667,0.007348,0.008933


,model_id,DockQ,fnat,iRMSD,LRMSD,F1
0,8cyj_H_D,0.010013,0.0,17.185648,56.051873,0.0
1,8b17_H_A,0.011878,0.0,23.229414,47.145746,0.0
2,8pyr_H_A,0.005221,0.0,29.755406,73.693801,0.0
3,7nxx_H_A,0.005102,0.0,28.870755,75.206119,0.0
4,7wg3_H_L_A,0.025549,0.0,16.580885,31.337497,0.0
...,...,...,...,...,...,...
204,7voa_H_A,0.014907,0.0,16.774452,43.493251,0.0
205,7ps6_H_L_E,0.005790,0.0,19.080668,79.766587,0.0
206,8hc3_H_L_A,0.001976,0.0,38.225250,127.986431,0.0
207,8emz_H_A,0.012062,0.0,20.171930,47.771999,0.0


,model_id,PTM,pLDDT
0,7r58_H_L_A,0.398,77.239
1,7sjs_H_L_A,0.422,73.066
2,7so5_H_L_A,0.570,69.460
3,7tp4_H_L_A,0.483,52.317
4,7u2d_H_L_A,0.420,50.850
...,...,...,...
205,7s11_I_M_D,0.473,72.541
206,7t5f_C_A,0.517,55.668
207,7t5f_E_D,0.558,58.641
208,7vnb_A_B,0.350,43.349


,model_id,pdb_id,Pdb_id_with_chains,ifpae_norm_ag,ifpae_norm_avg,ifplddt_ag,ifplddt_avg,PdockQ2 Antigen,PdockQ2_avg,DockQ,fnat,iRMSD,LRMSD,F1,PTM,pLDDT
0,7m1h_G_A,7m1h,7m1h_G_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,0.019704,0.0,14.955021,37.385247,0.0,0.558,53.819
1,8d9y_H_L_A,8d9y,8d9y_H_L_A,0.000000,0.623355,0.000000,56.687153,0.007348,0.358550,0.007914,0.0,23.838237,59.809277,0.0,0.641,77.071
2,7tcq_H_L_A,7tcq,7tcq_H_L_A,0.000000,0.624447,0.000000,56.916263,0.007348,0.366035,0.004690,0.0,27.329397,80.355779,0.0,0.721,80.437
3,7vaz_H_L_A,7vaz,7vaz_H_L_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,0.009424,0.0,29.855456,52.279414,0.0,0.413,66.695
4,8cyj_H_D,8cyj,8cyj_H_D,0.128239,0.129736,25.703333,37.791667,0.007997,0.008405,0.010013,0.0,17.185648,56.051873,0.0,0.387,46.035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,7xy8_H_L_A,7xy8,7xy8_H_L_A,0.000000,0.212582,0.000000,56.360631,0.007348,0.017305,0.005375,0.0,24.141662,76.237221,0.0,0.488,75.031
205,7l6v_D_A,7l6v,7l6v_D_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,0.023768,0.0,14.859742,33.285885,0.0,0.555,57.979
206,7l6v_F_A,7l6v,7l6v_F_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,0.007812,0.0,22.885468,60.818613,0.0,0.543,57.014
207,8tbq_H_L_A,8tbq,8tbq_H_L_A,0.000000,0.083764,0.000000,50.066667,0.007348,0.008933,0.005759,0.0,21.927116,75.188679,0.0,0.355,67.628


In [5]:
import os
import pandas as pd
import re

os.chdir("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/")

csv_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/newBM_33.csv"
df = pd.read_csv(csv_path)
complex_list=df["pdb_wchains"].to_list()
x = []

for i in complex_list:
    os.chdir(f"/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm/{i}_esm/")    
    log_file = "output.log" 

    ptm_score = None
    plddt_score = None

    with open(log_file, "r") as f:
        for line in f:
            match = re.search(r"PTM:\s*([\d.]+)\s*pLDDT:\s*([\d.]+)", line)
            if match:
                ptm_score = float(match.group(1))
                plddt_score = float(match.group(2))
                model = i
                break  # Stop after the first match

    df = pd.DataFrame({"model_id":[model],"PTM": [ptm_score], "pLDDT": [plddt_score]})
    x.append(df)
  
os.chdir("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm")

dfx = pd.concat(x, axis=0)
dfx.to_csv("esm_metrics_25models.csv", index=False)
dfx

,model_id,PTM,pLDDT
0,7r58_H_L_A,0.398,77.239
0,7sjs_H_L_A,0.422,73.066
0,7so5_H_L_A,0.570,69.460
0,7tp4_H_L_A,0.483,52.317
0,7u2d_H_L_A,0.420,50.850
...,...,...,...
0,7s11_I_M_D,0.473,72.541
0,7t5f_C_A,0.517,55.668
0,7t5f_E_D,0.558,58.641
0,7vnb_A_B,0.350,43.349


In [6]:
df4 = pd.merge(df3,dfx,on="model_id")
df4.to_csv("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/esm_pdockq2_dockq_metrics.csv")
df4

,model_id,pdb_id,Pdb_id_with_chains,ifpae_norm_ag,ifpae_norm_avg,ifplddt_ag,ifplddt_avg,PdockQ2 Antigen,PdockQ2_avg,ifpae_norm,...,prot_avg,PdockQ2,1 - Average PAE Interface Antigen,DockQ,fnat,iRMSD,LRMSD,F1,PTM,pLDDT
0,7m1h_G_A,7m1h,7m1h_G_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,NaN,...,NaN,NaN,NaN,0.019704,0.0,14.955021,37.385247,0.0,0.558,53.819
1,8d9y_H_L_A,8d9y,8d9y_H_L_A,0.000000,0.623355,0.000000,56.687153,0.007348,0.358550,NaN,...,NaN,NaN,NaN,0.007914,0.0,23.838237,59.809277,0.0,0.641,77.071
2,7tcq_H_L_A,7tcq,7tcq_H_L_A,0.000000,0.624447,0.000000,56.916263,0.007348,0.366035,NaN,...,NaN,NaN,NaN,0.004690,0.0,27.329397,80.355779,0.0,0.721,80.437
3,7vaz_H_L_A,7vaz,7vaz_H_L_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,NaN,...,NaN,NaN,NaN,0.009424,0.0,29.855456,52.279414,0.0,0.413,66.695
4,8cyj_H_D,8cyj,8cyj_H_D,0.128239,0.129736,25.703333,37.791667,0.007997,0.008405,NaN,...,NaN,NaN,NaN,0.010013,0.0,17.185648,56.051873,0.0,0.387,46.035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,7xy8_H_L_A,7xy8,7xy8_H_L_A,0.000000,0.212582,0.000000,56.360631,0.007348,0.017305,NaN,...,NaN,NaN,NaN,0.005375,0.0,24.141662,76.237221,0.0,0.488,75.031
414,7l6v_D_A,7l6v,7l6v_D_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,NaN,...,NaN,NaN,NaN,0.023768,0.0,14.859742,33.285885,0.0,0.555,57.979
415,7l6v_F_A,7l6v,7l6v_F_A,0.000000,0.000000,0.000000,0.000000,0.007348,0.007348,NaN,...,NaN,NaN,NaN,0.007812,0.0,22.885468,60.818613,0.0,0.543,57.014
416,8tbq_H_L_A,8tbq,8tbq_H_L_A,0.000000,0.083764,0.000000,50.066667,0.007348,0.008933,NaN,...,NaN,NaN,NaN,0.005759,0.0,21.927116,75.188679,0.0,0.355,67.628
